In [1]:
%pip install -q findspark


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# --- 0) Spark + module path bootstrap (run first) ---
import sys, importlib, findspark
findspark.init("/home/hduser/spark")  # change if Spark is elsewhere

# so Python can find your modules
sys.path.append("/home/student/de-ass/Task2ProcessData")

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("AQI-Process-Pipeline")
    .config("spark.sql.session.timeZone", "UTC")  # keep timestamps stable
    .getOrCreate()
)

print("Spark OK:", spark.version)


25/08/21 00:57:40 WARN Utils: Your hostname, Winni.localdomain resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/08/21 00:57:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/21 00:57:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark OK: 3.5.1


In [3]:
from preprocess.standardizer import Standardizer
from preprocess.fillers import Fillers
from preprocess.removeDuplicate import Deduplicator
from preprocess.rangeCheck import RangeGuards
from preprocess.imputers import Imputers
from preprocess.fill_AQI import AQI
from preprocess.enrich import Enricher


In [4]:
csv_path = "file:///home/student/de-ass/Task2ProcessData/noise.csv"

df_raw = (
    spark.read
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt")  # matches Standardizer.SCHEMA
    .schema(Standardizer.SCHEMA)
    .csv(csv_path)
)

df_raw.printSchema()
df_raw.show(19, truncate=False)


root
 |-- Date: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- AQI: double (nullable = true)
 |-- PM2.5 (µg/m³): double (nullable = true)
 |-- PM10 (µg/m³): double (nullable = true)
 |-- NO2 (ppb): double (nullable = true)
 |-- SO2 (ppb): double (nullable = true)
 |-- CO (ppm): double (nullable = true)
 |-- O3 (ppb): double (nullable = true)
 |-- Temperature (°C): double (nullable = true)
 |-- Humidity (%): double (nullable = true)
 |-- Wind Speed (m/s): double (nullable = true)
 |-- _corrupt: string (nullable = true)



+--------+-----------+-------+------+-------------+------------+---------+---------+--------+--------+----------------+------------+----------------+--------+
|Date    |City       |Country|AQI   |PM2.5 (µg/m³)|PM10 (µg/m³)|NO2 (ppb)|SO2 (ppb)|CO (ppm)|O3 (ppb)|Temperature (°C)|Humidity (%)|Wind Speed (m/s)|_corrupt|
+--------+-----------+-------+------+-------------+------------+---------+---------+--------+--------+----------------+------------+----------------+--------+
|1/1/2024|neW York   |USA    |-399.0|120.0        |182.9       |24.3     |26.0     |9.1     |153.3   |18.6            |-40.0       |13.2            |NULL    |
|1/1/2024|Los Angeles|USA    |280.0 |38.4         |46.9        |41.8     |34.7     |3.78    |190.7   |-2.2            |59.0        |9.5             |NULL    |
|1/1/2024|Delhi      |INDIA  |187.0 |-76.2        |226.8       |46.9     |17.2     |-1.02   |68.4    |9.9             |55.0        |3.3             |NULL    |
|1/1/2024|NULL       |FrancE |170.0 |217.6    

In [5]:
df_std = Standardizer.apply(df_raw).cache()
df_std.show(19, truncate=False)   # show all 19 rows without cutting text


+-------------------+-----------+-------+------+---------+---------+-------+-------+------+------+------+------------+-------+--------+
|ts                 |city       |country|AQI   |pm25_ugm3|pm10_ugm3|no2_ppb|so2_ppb|co_ppm|o3_ppb|temp_c|humidity_pct|wind_ms|_corrupt|
+-------------------+-----------+-------+------+---------+---------+-------+-------+------+------+------+------------+-------+--------+
|2024-01-01 00:00:00|New York   |USA    |-399.0|120.0    |182.9    |24.3   |26.0   |9.1   |153.3 |18.6  |-40.0       |13.2   |NULL    |
|2024-01-01 00:00:00|Los Angeles|USA    |280.0 |38.4     |46.9     |41.8   |34.7   |3.78  |190.7 |-2.2  |59.0        |9.5    |NULL    |
|2024-01-01 00:00:00|Delhi      |India  |187.0 |-76.2    |226.8    |46.9   |17.2   |-1.02 |68.4  |9.9   |55.0        |3.3    |NULL    |
|2024-01-01 00:00:00|NULL       |France |170.0 |217.6    |278.0    |21.7   |45.2   |1.77  |-125.2|37.4  |67.0        |1.4    |NULL    |
|2024-01-01 00:00:00|São Paulo  |Brazil |NULL  |

In [6]:
df_loc = Fillers.fill_city_country(df_std).cache()
df_loc.show(19, truncate=False)


+-------------------+-----------+-------+------+---------+---------+-------+-------+------+------+------+------------+-------+--------+
|ts                 |city       |country|AQI   |pm25_ugm3|pm10_ugm3|no2_ppb|so2_ppb|co_ppm|o3_ppb|temp_c|humidity_pct|wind_ms|_corrupt|
+-------------------+-----------+-------+------+---------+---------+-------+-------+------+------+------+------------+-------+--------+
|2024-01-01 00:00:00|New York   |USA    |-399.0|120.0    |182.9    |24.3   |26.0   |9.1   |153.3 |18.6  |-40.0       |13.2   |NULL    |
|2024-01-01 00:00:00|Los Angeles|USA    |280.0 |38.4     |46.9     |41.8   |34.7   |3.78  |190.7 |-2.2  |59.0        |9.5    |NULL    |
|2024-01-01 00:00:00|Delhi      |India  |187.0 |-76.2    |226.8    |46.9   |17.2   |-1.02 |68.4  |9.9   |55.0        |3.3    |NULL    |
|2024-01-01 00:00:00|Paris      |France |170.0 |217.6    |278.0    |21.7   |45.2   |1.77  |-125.2|37.4  |67.0        |1.4    |NULL    |
|2024-01-01 00:00:00|São Paulo  |Brazil |NULL  |

In [7]:
df_dates = Fillers.fill_missing_date(df_loc).cache()
df_dates.show(19, truncate=False)


+-------------------+-----------+-------+------+---------+---------+-------+-------+------+------+------+------------+-------+--------+
|ts                 |city       |country|AQI   |pm25_ugm3|pm10_ugm3|no2_ppb|so2_ppb|co_ppm|o3_ppb|temp_c|humidity_pct|wind_ms|_corrupt|
+-------------------+-----------+-------+------+---------+---------+-------+-------+------+------+------+------------+-------+--------+
|2024-01-01 00:00:00|New York   |USA    |-399.0|120.0    |182.9    |24.3   |26.0   |9.1   |153.3 |18.6  |-40.0       |13.2   |NULL    |
|2024-01-02 00:00:00|New York   |USA    |32.0  |69.7     |NULL     |28.1   |43.2   |8.04  |114.3 |38.2  |73.0        |1.7    |NULL    |
|2024-01-03 00:00:00|New York   |USA    |251.0 |99.0     |102.8    |85.5   |NULL   |2.28  |160.9 |-7.3  |60.0        |12.3   |NULL    |
|2024-01-03 00:00:00|NULL       |NULL   |37.0  |103.1    |252.0    |89.2   |42.1   |8.81  |52.7  |9.9   |68.0        |8.2    |NULL    |
|2024-01-01 00:00:00|Cairo      |Egypt  |241.0 |

In [8]:
df_duplicate = Deduplicator.dedup_daily(df_dates).cache()
df_duplicate.show(19, truncate=False)

25/08/21 00:57:54 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------------------+-----------+-------+------+---------+---------+-------+-------+------+------+------+------------+-------+--------+----------+
|ts                 |city       |country|AQI   |pm25_ugm3|pm10_ugm3|no2_ppb|so2_ppb|co_ppm|o3_ppb|temp_c|humidity_pct|wind_ms|_corrupt|event_date|
+-------------------+-----------+-------+------+---------+---------+-------+-------+------+------+------+------------+-------+--------+----------+
|2024-01-01 00:00:00|New York   |USA    |-399.0|120.0    |182.9    |24.3   |26.0   |9.1   |153.3 |18.6  |-40.0       |13.2   |NULL    |2024-01-01|
|2024-01-02 00:00:00|New York   |USA    |32.0  |69.7     |NULL     |28.1   |43.2   |8.04  |114.3 |38.2  |73.0        |1.7    |NULL    |2024-01-02|
|2024-01-03 00:00:00|New York   |USA    |251.0 |99.0     |102.8    |85.5   |NULL   |2.28  |160.9 |-7.3  |60.0        |12.3   |NULL    |2024-01-03|
|2024-01-03 00:00:00|NULL       |NULL   |37.0  |103.1    |252.0    |89.2   |42.1   |8.81  |52.7  |9.9   |68.0        |

In [9]:
df_checked = RangeGuards.column_ranges(df_duplicate, mode="nullify")   # or mode="clip"
df_checked.show(19, truncate=False)

+-------------------+-----------+-------+-----+---------+---------+-------+-------+------+------+------+------------+-------+--------+----------+
|ts                 |city       |country|AQI  |pm25_ugm3|pm10_ugm3|no2_ppb|so2_ppb|co_ppm|o3_ppb|temp_c|humidity_pct|wind_ms|_corrupt|event_date|
+-------------------+-----------+-------+-----+---------+---------+-------+-------+------+------+------+------------+-------+--------+----------+
|2024-01-01 00:00:00|New York   |USA    |NULL |120.0    |182.9    |24.3   |26.0   |9.1   |153.3 |18.6  |NULL        |13.2   |NULL    |2024-01-01|
|2024-01-02 00:00:00|New York   |USA    |32.0 |69.7     |NULL     |28.1   |43.2   |8.04  |114.3 |38.2  |73.0        |1.7    |NULL    |2024-01-02|
|2024-01-03 00:00:00|New York   |USA    |251.0|99.0     |102.8    |85.5   |NULL   |2.28  |160.9 |-7.3  |60.0        |12.3   |NULL    |2024-01-03|
|2024-01-03 00:00:00|NULL       |NULL   |37.0 |103.1    |252.0    |89.2   |42.1   |8.81  |52.7  |9.9   |68.0        |8.2    

In [10]:
df_imp = Imputers.numeric(df_checked).cache()
df_imp.show(19, truncate=False)

+-------------------+-----------+-------+-----+---------+---------+-------+-------+------+------+------+------------+-------+--------+----------+
|ts                 |city       |country|AQI  |pm25_ugm3|pm10_ugm3|no2_ppb|so2_ppb|co_ppm|o3_ppb|temp_c|humidity_pct|wind_ms|_corrupt|event_date|
+-------------------+-----------+-------+-----+---------+---------+-------+-------+------+------+------+------------+-------+--------+----------+
|2024-01-01 00:00:00|New York   |USA    |NULL |120.0    |182.9    |24.3   |26.0   |9.1   |153.3 |18.6  |60.0        |13.2   |NULL    |2024-01-01|
|2024-01-02 00:00:00|New York   |USA    |32.0 |69.7     |102.8    |28.1   |43.2   |8.04  |114.3 |38.2  |73.0        |1.7    |NULL    |2024-01-02|
|2024-01-03 00:00:00|New York   |USA    |251.0|99.0     |102.8    |85.5   |34.7   |2.28  |160.9 |-7.3  |60.0        |12.3   |NULL    |2024-01-03|
|2024-01-03 00:00:00|NULL       |NULL   |37.0 |103.1    |252.0    |89.2   |42.1   |8.81  |52.7  |9.9   |68.0        |8.2    

In [11]:
df_aqi = AQI.impute_aqi(df_imp).cache()
df_aqi.show(19, truncate=False)


+-------------------+-----------+-------+-----------------+---------+---------+-------+-------+------+------+------+------------+-------+--------+----------+
|ts                 |city       |country|AQI              |pm25_ugm3|pm10_ugm3|no2_ppb|so2_ppb|co_ppm|o3_ppb|temp_c|humidity_pct|wind_ms|_corrupt|event_date|
+-------------------+-----------+-------+-----------------+---------+---------+-------+-------+------+------+------+------------+-------+--------+----------+
|2024-01-01 00:00:00|New York   |USA    |250.8159574468085|120.0    |182.9    |24.3   |26.0   |9.1   |153.3 |18.6  |60.0        |13.2   |NULL    |2024-01-01|
|2024-01-02 00:00:00|New York   |USA    |32.0             |69.7     |102.8    |28.1   |43.2   |8.04  |114.3 |38.2  |73.0        |1.7    |NULL    |2024-01-02|
|2024-01-03 00:00:00|New York   |USA    |251.0            |99.0     |102.8    |85.5   |34.7   |2.28  |160.9 |-7.3  |60.0        |12.3   |NULL    |2024-01-03|
|2024-01-03 00:00:00|NULL       |NULL   |37.0       

In [12]:
df_enriched = Enricher.apply(df_aqi, buckets=True, who_flags=True).cache()
df_enriched.show(19, truncate=False)

+-------------------+-----------+-------+-----------------+---------+---------+-------+-------+------+------+------+------------+-------+--------+----------+----+-----+---+-----------+----------+------+-------------------+------------+---------------+---------------+--------------+--------------+-------------+-------------+----------------+----------------+---------------+---------------+--------------+--------------+-----------------+
|ts                 |city       |country|AQI              |pm25_ugm3|pm10_ugm3|no2_ppb|so2_ppb|co_ppm|o3_ppb|temp_c|humidity_pct|wind_ms|_corrupt|event_date|year|month|day|day_of_week|hemisphere|season|pm_ratio           |aqi_category|aqi_bucket_pm25|aqi_bucket_pm10|aqi_bucket_no2|aqi_bucket_so2|aqi_bucket_co|aqi_bucket_o3|pm25_exceeds_who|pm10_exceeds_who|no2_exceeds_who|so2_exceeds_who|co_exceeds_who|o3_exceeds_who|who_compliant_all|
+-------------------+-----------+-------+-----------------+---------+---------+-------+-------+------+------+------+----

In [15]:
from validate.validator_pipeline import ValidatorPipeline 

df_valid, df_invalid, df_invalid_labeled = ValidatorPipeline.run(df_enriched)

print("Valid rows :", df_valid.count())
print("Invalid rows:", df_invalid.count())


Valid rows : 17
Invalid rows: 1


In [16]:
df_invalid_labeled.groupBy("error_code").count() \
    .orderBy(F.desc("count")).show(truncate=False)


+----------+-----+
|error_code|count|
+----------+-----+
|CITY_EMPTY|1    |
+----------+-----+

